# RegimeLens — HMM labels and next-day classification (historical)

**Trương Minh Hoàng · MSSV 25410008**

**ARCHIVED_SOURCE — source review, not a fresh benchmark.** Bản học tập lịch sử được giữ để giải thích quá trình làm bài. Output cũ và thông tin môi trường riêng đã được bỏ; xem README/AUDIT của dự án trước khi chạy. Historical calculations and known limitations are documented separately.

In [ ]:
# ============================================================
# CELL 1: THIẾT LẬP
# ============================================================

# Mount Google Drive
# Optional Colab mount must be configured by the user.

# Cài thư viện cần thiết cho Gaussian Hidden Markov Model
# Install research/requirements.txt before running.

# Import thư viện xử lý dữ liệu
import os
import warnings
import numpy as np
import pandas as pd

# Import thư viện trực quan hóa
import matplotlib.pyplot as plt
import seaborn as sns

# Import thư viện machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Import Gaussian Hidden Markov Model
from hmmlearn.hmm import GaussianHMM

# Lưu model
import joblib


# Tắt warning không cần thiết để notebook dễ đọc hơn
warnings.filterwarnings("ignore")

# Cấu hình hiển thị dữ liệu
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Khai báo đường dẫn chính
base_path = os.environ.get("REGIMELENS_DATA_DIR", "data/local").rstrip("/") + "/"

# Khai báo đường dẫn file dữ liệu
stock_file = base_path + "SP500_Historical_Data.csv"
sector_file = base_path + "SP500.csv"
vix_file = base_path + "VIXCLS.csv"

# Khai báo thư mục lưu output
output_path = base_path + "outputs/"
chart_path = base_path + "charts/"
model_path = base_path + "models/"

# Tạo thư mục nếu chưa tồn tại
os.makedirs(output_path, exist_ok=True)
os.makedirs(chart_path, exist_ok=True)
os.makedirs(model_path, exist_ok=True)

print("Thiết lập môi trường hoàn tất thành công.")
print("Đường dẫn chính:", base_path)
print("Đường dẫn output:", output_path)
print("Đường dẫn chart:", chart_path)
print("Đường dẫn model:", model_path)

In [ ]:
# ============================================================
# CELL 2: LOAD DATA
# ============================================================

# Đọc dữ liệu cổ phiếu S&P 500
stock_df = pd.read_csv(stock_file)

# Đọc dữ liệu sector / company metadata
sector_df = pd.read_csv(sector_file)

# Đọc dữ liệu VIX
vix_df = pd.read_csv(vix_file)

print("Đọc dữ liệu hoàn tất thành công.")

print("\nKích thước stock data:", stock_df.shape)
print("Kích thước sector data:", sector_df.shape)
print("Kích thước VIX data:", vix_df.shape)

print("\nCác cột của stock data:")
print(stock_df.columns.tolist())

print("\nCác cột của sector data:")
print(sector_df.columns.tolist())

print("\nCác cột của VIX data:")
print(vix_df.columns.tolist())

print("\n5 dòng đầu của stock data:")
display(stock_df.head())

print("\n5 dòng đầu của sector data:")
display(sector_df.head())

print("\n5 dòng đầu của VIX data:")
display(vix_df.head())

In [ ]:
# ============================================================
# CELL 3: TỔNG QUAN DỮ LIỆU
# ============================================================

print("========== THÔNG TIN STOCK DATA ==========")
stock_df.info()

print("\n========== THÔNG TIN SECTOR DATA ==========")
sector_df.info()

print("\n========== THÔNG TIN VIX DATA ==========")
vix_df.info()


print("\n========== MISSING VALUES ==========")

print("\nMissing values của stock data:")
display(stock_df.isnull().sum().sort_values(ascending=False))

print("\nMissing values của sector data:")
display(sector_df.isnull().sum().sort_values(ascending=False))

print("\nMissing values của VIX data:")
display(vix_df.isnull().sum().sort_values(ascending=False))


print("\n========== DÒNG TRÙNG LẶP ==========")

stock_duplicates = stock_df.duplicated().sum()
sector_duplicates = sector_df.duplicated().sum()
vix_duplicates = vix_df.duplicated().sum()

print("Số dòng trùng lặp của stock data:", stock_duplicates)
print("Số dòng trùng lặp của sector data:", sector_duplicates)
print("Số dòng trùng lặp của VIX data:", vix_duplicates)


print("\n========== KHOẢNG THỜI GIAN ==========")

# Kiểm tra khoảng thời gian của stock_df nếu có cột Date
if "Date" in stock_df.columns:
    stock_date_temp = pd.to_datetime(stock_df["Date"], errors="coerce")
    print("Khoảng thời gian của stock data:")
    print("Ngày bắt đầu:", stock_date_temp.min())
    print("Ngày kết thúc:", stock_date_temp.max())
else:
    print("Không tìm thấy cột 'Date' trong stock_df.")

# Kiểm tra khoảng thời gian của vix_df nếu có cột observation_date
if "observation_date" in vix_df.columns:
    vix_date_temp = pd.to_datetime(vix_df["observation_date"], errors="coerce")
    print("\nKhoảng thời gian của VIX data:")
    print("Ngày bắt đầu:", vix_date_temp.min())
    print("Ngày kết thúc:", vix_date_temp.max())
else:
    print("Không tìm thấy cột 'observation_date' trong vix_df.")


print("\n========== TỔNG QUAN TICKER ==========")

# Kiểm tra số lượng ticker nếu có cột Ticker
if "Ticker" in stock_df.columns:
    number_of_tickers = stock_df["Ticker"].nunique()
    print("Số lượng ticker duy nhất:", number_of_tickers)

    print("\nTop 10 ticker theo số dòng:")
    display(stock_df["Ticker"].value_counts().head(10))
else:
    print("Không tìm thấy cột 'Ticker' trong stock_df.")


print("\n========== TỔNG QUAN SECTOR ==========")

# Kiểm tra số lượng sector nếu có cột GICS Sector
if "GICS Sector" in sector_df.columns:
    number_of_sectors = sector_df["GICS Sector"].nunique()
    print("Số lượng sector duy nhất:", number_of_sectors)

    print("\nPhân phối sector:")
    display(sector_df["GICS Sector"].value_counts())
else:
    print("Không tìm thấy cột 'GICS Sector' trong sector_df.")


print("\nTổng quan dữ liệu hoàn tất thành công.")

In [ ]:
# ============================================================
# CELL 4: LÀM SẠCH DỮ LIỆU
# ============================================================

print("Bắt đầu làm sạch dữ liệu...")


# ------------------------------------------------------------
# 1. Chuẩn hóa tên cột
# ------------------------------------------------------------

stock_df = stock_df.rename(columns={
    "Ticker": "ticker",
    "Date": "date",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Adj Close": "adj_close",
    "Volume": "volume"
})

sector_df = sector_df.rename(columns={
    "Symbol": "ticker",
    "Security": "security",
    "GICS Sector": "gics_sector",
    "GICS Sub-Industry": "gics_sub_industry"
})

vix_df = vix_df.rename(columns={
    "observation_date": "date",
    "VIXCLS": "vix_close"
})


print("Chuẩn hóa tên cột hoàn tất thành công.")

print("\nCác cột stock data:")
print(stock_df.columns.tolist())

print("\nCác cột sector data:")
print(sector_df.columns.tolist())

print("\nCác cột VIX data:")
print(vix_df.columns.tolist())


# ------------------------------------------------------------
# 2. Đổi cột ngày sang datetime
# ------------------------------------------------------------

stock_df["date"] = pd.to_datetime(stock_df["date"], errors="coerce")
vix_df["date"] = pd.to_datetime(vix_df["date"], errors="coerce")


# ------------------------------------------------------------
# 3. Đổi numeric columns sang dạng số
# ------------------------------------------------------------

stock_numeric_cols = [
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume"
]

for col in stock_numeric_cols:
    if col in stock_df.columns:
        stock_df[col] = pd.to_numeric(stock_df[col], errors="coerce")

vix_df["vix_close"] = pd.to_numeric(vix_df["vix_close"], errors="coerce")


# ------------------------------------------------------------
# 4. Xóa các dòng trùng lặp
# ------------------------------------------------------------

stock_before = stock_df.shape[0]
sector_before = sector_df.shape[0]
vix_before = vix_df.shape[0]

stock_df = stock_df.drop_duplicates()
sector_df = sector_df.drop_duplicates()
vix_df = vix_df.drop_duplicates()

print("\nDuplicate rows removed:")
print("Stock removed:", stock_before - stock_df.shape[0])
print("Sector removed:", sector_before - sector_df.shape[0])
print("VIX removed:", vix_before - vix_df.shape[0])


# ------------------------------------------------------------
# 5. Xử lý missing values quan trọng
# ------------------------------------------------------------

# Xóa dòng không có ticker hoặc date vì không thể dùng để phân tích theo thời gian
stock_df = stock_df.dropna(subset=["ticker", "date"])
sector_df = sector_df.dropna(subset=["ticker"])
vix_df = vix_df.dropna(subset=["date"])

# Sắp xếp dữ liệu trước khi fill missing theo từng ticker
stock_df = stock_df.sort_values(["ticker", "date"]).reset_index(drop=True)
vix_df = vix_df.sort_values("date").reset_index(drop=True)

# Nếu adj_close thiếu thì dùng close thay thế
if "adj_close" in stock_df.columns and "close" in stock_df.columns:
    stock_df["adj_close"] = stock_df["adj_close"].fillna(stock_df["close"])

# Fill missing numeric theo từng ticker
stock_df[stock_numeric_cols] = (
    stock_df
    .groupby("ticker")[stock_numeric_cols]
    .ffill()
    .bfill()
)

# Sau khi fill, nếu vẫn thiếu các cột giá quan trọng thì xóa
stock_df = stock_df.dropna(subset=["open", "high", "low", "close", "adj_close"])

# Volume nếu vẫn thiếu thì cho bằng 0
stock_df["volume"] = stock_df["volume"].fillna(0)

# VIX fill missing theo thời gian
# Chỉ dùng ffill, không dùng bfill để tránh lấy dữ liệu tương lai điền ngược về quá khứ
vix_df["vix_close"] = vix_df["vix_close"].ffill()

# ------------------------------------------------------------
# Filter stock data theo khoảng thời gian VIX có thật
# ------------------------------------------------------------

min_vix_date = vix_df["date"].min()
max_vix_date = vix_df["date"].max()

print("\nVIX available date range:")
print("Start:", min_vix_date)
print("End:", max_vix_date)

stock_before_vix_filter = stock_df.shape[0]

stock_df = stock_df[
    (stock_df["date"] >= min_vix_date) &
    (stock_df["date"] <= max_vix_date)
].reset_index(drop=True)

print("\nStock rows before VIX date filter:", stock_before_vix_filter)
print("Stock rows after VIX date filter:", stock_df.shape[0])

# Sector metadata nếu thiếu thì gán Unknown
sector_text_cols = ["security", "gics_sector", "gics_sub_industry"]

for col in sector_text_cols:
    if col in sector_df.columns:
        sector_df[col] = sector_df[col].fillna("Unknown")


# ------------------------------------------------------------
# 6. Tối ưu datatype để nhẹ hơn trên Colab
# ------------------------------------------------------------

# Các cột text/categorical
stock_df["ticker"] = stock_df["ticker"].astype("category")
sector_df["ticker"] = sector_df["ticker"].astype("category")

for col in sector_text_cols:
    if col in sector_df.columns:
        sector_df[col] = sector_df[col].astype("category")

# Các cột numeric dùng float32 để tiết kiệm RAM
for col in stock_numeric_cols:
    stock_df[col] = stock_df[col].astype("float32")

vix_df["vix_close"] = vix_df["vix_close"].astype("float32")


# ------------------------------------------------------------
# 7. Kiểm tra sau cleaning
# ------------------------------------------------------------

print("\n========== KÍCH THƯỚC DỮ LIỆU SAU CLEANING ==========")
print("Kích thước stock data sau cleaning:", stock_df.shape)
print("Kích thước sector data sau cleaning:", sector_df.shape)
print("Kích thước VIX data sau cleaning:", vix_df.shape)

print("\n========== THÔNG TIN STOCK DATA SAU CLEANING ==========")
stock_df.info()

print("\n========== MISSING VALUES SAU CLEANING ==========")

print("\nMissing values của stock data:")
display(stock_df.isnull().sum().sort_values(ascending=False))

print("\nMissing values của sector data:")
display(sector_df.isnull().sum().sort_values(ascending=False))

print("\nMissing values của VIX data:")
display(vix_df.isnull().sum().sort_values(ascending=False))


# ------------------------------------------------------------
# 8. Lưu stock_clean.parquet
# ------------------------------------------------------------

stock_clean_file = output_path + "stock_clean.parquet"

stock_df.to_parquet(stock_clean_file, index=False)

print("\nStock data sau cleaning đã lưu tại:")
print(stock_clean_file)

print("\nLàm sạch dữ liệu hoàn tất thành công.")

In [ ]:
# ============================================================
# CELL 5: JOIN DATA
# ============================================================

print("Bắt đầu join dữ liệu...")


# ------------------------------------------------------------
# 1. Chuẩn hóa key join
# ------------------------------------------------------------

# Chuyển ticker về string, strip khoảng trắng, viết hoa để join an toàn hơn
stock_df["ticker"] = stock_df["ticker"].astype(str).str.strip().str.upper()
sector_df["ticker"] = sector_df["ticker"].astype(str).str.strip().str.upper()

# Đảm bảo date của stock_df và vix_df đều là datetime
stock_df["date"] = pd.to_datetime(stock_df["date"], errors="coerce")
vix_df["date"] = pd.to_datetime(vix_df["date"], errors="coerce")


# ------------------------------------------------------------
# 2. Xử lý duplicate trong sector và VIX trước khi join
# ------------------------------------------------------------

# Mỗi ticker chỉ nên có một dòng metadata
sector_unique = (
    sector_df
    .sort_values("ticker")
    .drop_duplicates(subset=["ticker"], keep="first")
    .reset_index(drop=True)
)

# Mỗi ngày chỉ nên có một giá trị VIX
vix_unique = (
    vix_df
    .sort_values("date")
    .drop_duplicates(subset=["date"], keep="first")
    .reset_index(drop=True)
)


print("Kích thước sector unique:", sector_unique.shape)
print("Kích thước VIX unique:", vix_unique.shape)


# ------------------------------------------------------------
# 3. Join stock data với sector metadata
# ------------------------------------------------------------

stock_joined_df = stock_df.merge(
    sector_unique,
    on="ticker",
    how="left"
)

print("\nSau khi join sector metadata:")
print("Kích thước stock data sau join:", stock_joined_df.shape)


# ------------------------------------------------------------
# 4. Join stock data với VIX theo date
# ------------------------------------------------------------

stock_joined_df = stock_joined_df.merge(
    vix_unique[["date", "vix_close"]],
    on="date",
    how="left"
)

print("\nSau khi join VIX:")
print("Kích thước stock data sau join:", stock_joined_df.shape)


# ------------------------------------------------------------
# 5. Xử lý missing sau join
# ------------------------------------------------------------

# Nếu ticker không match được với sector metadata thì gán Unknown
sector_cols = ["security", "gics_sector", "gics_sub_industry"]

for col in sector_cols:
    if col in stock_joined_df.columns:
        # Thêm 'Unknown' vào các categories trước khi fillna
        if "Unknown" not in stock_joined_df[col].cat.categories:
            stock_joined_df[col] = stock_joined_df[col].cat.add_categories("Unknown")
        stock_joined_df[col] = stock_joined_df[col].fillna("Unknown")

# VIX là dữ liệu market-level, chỉ fill tiến theo thời gian
# Không dùng bfill để tránh lấy dữ liệu tương lai điền ngược về quá khứ
stock_joined_df = stock_joined_df.sort_values(["date", "ticker"]).reset_index(drop=True)

stock_joined_df["vix_close"] = stock_joined_df["vix_close"].ffill()


# ------------------------------------------------------------
# 6. Tối ưu datatype sau join
# ------------------------------------------------------------

stock_joined_df["ticker"] = stock_joined_df["ticker"].astype("category")

for col in sector_cols:
    if col in stock_joined_df.columns:
        stock_joined_df[col] = stock_joined_df[col].astype("category")

stock_joined_df["vix_close"] = stock_joined_df["vix_close"].astype("float32")


# ------------------------------------------------------------
# 7. Kiểm tra dữ liệu sau join
# ------------------------------------------------------------

print("\n========== TỔNG QUAN DỮ LIỆU SAU JOIN ==========")
print("Kích thước dữ liệu sau join:", stock_joined_df.shape)

print("\nCác cột sau join:")
print(stock_joined_df.columns.tolist())

print("\nMissing values sau join:")
display(stock_joined_df.isnull().sum().sort_values(ascending=False))

print("\nSố lượng ticker duy nhất sau join:")
print(stock_joined_df["ticker"].nunique())

print("\nPhân phối sector sau join:")
display(stock_joined_df["gics_sector"].value_counts())

print("\nThống kê VIX sau join:")
display(stock_joined_df["vix_close"].describe())

print("\nMẫu dữ liệu sau join:")
display(stock_joined_df.head(10))

print("\nJoin dữ liệu hoàn tất thành công.")

In [ ]:
# ============================================================
# CELL 6: FEATURE ENGINEERING AT STOCK LEVEL
# ============================================================

print("Bắt đầu feature engineering ở cấp cổ phiếu...")


# ------------------------------------------------------------
# 1. Tạo bản copy từ dữ liệu đã join
# ------------------------------------------------------------

stock_features_df = stock_joined_df.copy()

# Sắp xếp theo từng ticker và thời gian
stock_features_df = stock_features_df.sort_values(["ticker", "date"]).reset_index(drop=True)


# ------------------------------------------------------------
# 2. Tạo group theo từng cổ phiếu
# ------------------------------------------------------------

ticker_group = stock_features_df.groupby("ticker", observed=True)


# ------------------------------------------------------------
# 3. Tạo return features
# ------------------------------------------------------------

# Daily return: tỷ suất sinh lời theo ngày
stock_features_df["daily_return"] = ticker_group["adj_close"].pct_change()

# Log return: log(adj_close_t / adj_close_t-1)
stock_features_df["log_return"] = np.log(
    stock_features_df["adj_close"] / ticker_group["adj_close"].shift(1)
)


# ------------------------------------------------------------
# 4. Tạo rolling volatility 20 ngày
# ------------------------------------------------------------

# Rolling volatility 20 ngày từ daily_return
# Nhân sqrt(252) để annualize volatility
stock_features_df["rolling_volatility_20"] = (
    ticker_group["daily_return"]
    .transform(lambda x: x.rolling(window=20, min_periods=20).std())
    * np.sqrt(252)
)


# ------------------------------------------------------------
# 5. Tạo moving average features
# ------------------------------------------------------------

stock_features_df["ma20"] = ticker_group["adj_close"].transform(
    lambda x: x.rolling(window=20, min_periods=20).mean()
)

stock_features_df["ma50"] = ticker_group["adj_close"].transform(
    lambda x: x.rolling(window=50, min_periods=50).mean()
)

stock_features_df["ma200"] = ticker_group["adj_close"].transform(
    lambda x: x.rolling(window=200, min_periods=200).mean()
)


# ------------------------------------------------------------
# 6. Tạo volume change
# ------------------------------------------------------------

stock_features_df["volume_change"] = ticker_group["volume"].pct_change()


# ------------------------------------------------------------
# 7. Tạo biến cổ phiếu nằm trên MA50 / MA200
# ------------------------------------------------------------

# Nếu MA chưa đủ dữ liệu thì để NaN, không ép thành 0
stock_features_df["above_ma50"] = np.where(
    stock_features_df["ma50"].notna(),
    (stock_features_df["adj_close"] > stock_features_df["ma50"]).astype(int),
    np.nan
)

stock_features_df["above_ma200"] = np.where(
    stock_features_df["ma200"].notna(),
    (stock_features_df["adj_close"] > stock_features_df["ma200"]).astype(int),
    np.nan
)


# ------------------------------------------------------------
# 8. Xử lý inf values sinh ra do chia cho 0
# ------------------------------------------------------------

stock_features_df = stock_features_df.replace([np.inf, -np.inf], np.nan)


# ------------------------------------------------------------
# 9. Tối ưu datatype cho các feature mới
# ------------------------------------------------------------

feature_cols = [
    "daily_return",
    "log_return",
    "rolling_volatility_20",
    "ma20",
    "ma50",
    "ma200",
    "volume_change",
    "above_ma50",
    "above_ma200"
]

for col in feature_cols:
    stock_features_df[col] = stock_features_df[col].astype("float32")


# ------------------------------------------------------------
# 10. Kiểm tra dữ liệu sau feature engineering
# ------------------------------------------------------------

print("\n========== TỔNG QUAN STOCK FEATURES ==========")
print("Kích thước stock features:", stock_features_df.shape)

print("\nCác feature mới:")
print(feature_cols)

print("\nMissing values trong các feature mới:")
display(stock_features_df[feature_cols].isnull().sum().sort_values(ascending=False))

print("\nMẫu stock features:")
display(stock_features_df.head(10))

print("\nThống kê mô tả stock features:")
display(stock_features_df[feature_cols].describe())


# ------------------------------------------------------------
# 11. Lưu stock_features.parquet
# ------------------------------------------------------------

stock_features_file = output_path + "stock_features.parquet"

stock_features_df.to_parquet(stock_features_file, index=False)

print("\nStock features đã lưu tại:")
print(stock_features_file)

print("\nFeature engineering ở cấp cổ phiếu hoàn tất thành công.")

In [ ]:
# ============================================================
# CELL 7: TỔNG HỢP MARKET-LEVEL
# ============================================================

print("Bắt đầu tổng hợp market-level...")


# ------------------------------------------------------------
# 1. Tạo bản copy từ stock-level features
# ------------------------------------------------------------

market_source_df = stock_features_df.copy()

# Đảm bảo dữ liệu được sắp xếp theo ngày
market_source_df = market_source_df.sort_values(["date", "ticker"]).reset_index(drop=True)


# ------------------------------------------------------------
# 2. Tạo các biến phụ để tính market breadth
# ------------------------------------------------------------

# Cổ phiếu tăng giá trong ngày
market_source_df["is_advance"] = np.where(
    market_source_df["daily_return"] > 0,
    1,
    0
)

# Cổ phiếu giảm giá trong ngày
market_source_df["is_decline"] = np.where(
    market_source_df["daily_return"] < 0,
    1,
    0
)

# Chỉ tính những dòng có daily_return hợp lệ
market_source_df["has_valid_return"] = np.where(
    market_source_df["daily_return"].notna(),
    1,
    0
)


# ------------------------------------------------------------
# 3. Aggregate các feature chính theo ngày
# ------------------------------------------------------------

market_agg_df = (
    market_source_df
    .groupby("date")
    .agg(
        market_return=("daily_return", "mean"),
        advance_count=("is_advance", "sum"),
        decline_count=("is_decline", "sum"),
        valid_return_count=("has_valid_return", "sum"),
        percent_stocks_above_ma50=("above_ma50", "mean"),
        percent_stocks_above_ma200=("above_ma200", "mean"),
        avg_volume_change=("volume_change", "mean"),
        vix_close=("vix_close", "mean"),
        number_of_stocks=("ticker", "nunique")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. Tạo advance_decline_ratio
# ------------------------------------------------------------

# Ở đây dùng tỷ lệ cổ phiếu tăng giá trên tổng số cổ phiếu có return hợp lệ
# Công thức này ổn định hơn advances / declines vì không bị vô cực khi decline_count = 0
market_agg_df["advance_decline_ratio"] = (
    market_agg_df["advance_count"] / market_agg_df["valid_return_count"]
)

# Xử lý trường hợp valid_return_count = 0
market_agg_df["advance_decline_ratio"] = market_agg_df["advance_decline_ratio"].replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# 5. Tạo market_volatility_20
# ------------------------------------------------------------

# Volatility 20 ngày của market_return
# Nhân sqrt(252) để annualize volatility
market_agg_df = market_agg_df.sort_values("date").reset_index(drop=True)

market_agg_df["market_volatility_20"] = (
    market_agg_df["market_return"]
    .rolling(window=20, min_periods=20)
    .std()
    * np.sqrt(252)
)


# ------------------------------------------------------------
# 6. Tạo sector_return theo từng ngành
# ------------------------------------------------------------

sector_return_df = (
    market_source_df
    .groupby(["date", "gics_sector"], observed=True)
    .agg(
        sector_return=("daily_return", "mean")
    )
    .reset_index()
)

sector_return_pivot_df = sector_return_df.pivot(
    index="date",
    columns="gics_sector",
    values="sector_return"
).reset_index()

# Chuẩn hóa tên cột sector return
sector_return_pivot_df.columns = [
    "date" if col == "date"
    else "sector_return_" + str(col).lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("&", "and")
        .replace("/", "_")
    for col in sector_return_pivot_df.columns
]


# ------------------------------------------------------------
# 7. Join sector_return vào market-level dataset
# ------------------------------------------------------------

market_level_df = market_agg_df.merge(
    sector_return_pivot_df,
    on="date",
    how="left"
)


# ------------------------------------------------------------
# 8. Sắp xếp lại thứ tự cột cho dễ đọc
# ------------------------------------------------------------

main_market_cols = [
    "date",
    "market_return",
    "market_volatility_20",
    "advance_decline_ratio",
    "percent_stocks_above_ma50",
    "percent_stocks_above_ma200",
    "avg_volume_change",
    "vix_close",
    "number_of_stocks",
    "advance_count",
    "decline_count",
    "valid_return_count"
]

sector_return_cols = [
    col for col in market_level_df.columns
    if col.startswith("sector_return_")
]

market_level_df = market_level_df[main_market_cols + sector_return_cols]


# ------------------------------------------------------------
# 9. Xử lý missing và inf values
# ------------------------------------------------------------

market_level_df = market_level_df.replace([np.inf, -np.inf], np.nan)

# Các dòng đầu có thể thiếu market_volatility_20 do rolling 20 ngày
# Không xóa ở đây, vì Cell 8 sẽ dùng HMM và xử lý missing trước khi train HMM
market_level_df = market_level_df.sort_values("date").reset_index(drop=True)


# ------------------------------------------------------------
# 10. Tối ưu datatype
# ------------------------------------------------------------

float_cols = market_level_df.select_dtypes(include=["float64", "float32"]).columns

for col in float_cols:
    market_level_df[col] = market_level_df[col].astype("float32")

int_cols = [
    "number_of_stocks",
    "advance_count",
    "decline_count",
    "valid_return_count"
]

for col in int_cols:
    market_level_df[col] = market_level_df[col].astype("int32")


# ------------------------------------------------------------
# 11. Kiểm tra market-level dataset
# ------------------------------------------------------------

print("\n========== TỔNG QUAN MARKET-LEVEL DATA ==========")
print("Kích thước market-level data:", market_level_df.shape)

print("\nCác cột market-level:")
print(market_level_df.columns.tolist())

print("\nKhoảng thời gian:")
print("Ngày bắt đầu:", market_level_df["date"].min())
print("Ngày kết thúc:", market_level_df["date"].max())

print("\nMissing values:")
display(market_level_df.isnull().sum().sort_values(ascending=False))

print("\nMẫu market-level data:")
display(market_level_df.head(10))

print("\nThống kê mô tả market-level:")
display(market_level_df.describe())


# ------------------------------------------------------------
# 12. Lưu market_level_dataset.parquet
# ------------------------------------------------------------

market_level_file = output_path + "market_level_dataset.parquet"

market_level_df.to_parquet(market_level_file, index=False)

print("\nMarket-level dataset đã lưu tại:")
print(market_level_file)

print("\nTổng hợp market-level hoàn tất thành công.")

In [ ]:
# ============================================================
# CELL 8: GÁN NHÃN REGIME BẰNG GAUSSIAN HIDDEN MARKOV MODEL
# ============================================================

print("Bắt đầu gán nhãn regime bằng Gaussian Hidden Markov Model...")


# ------------------------------------------------------------
# 1. Tạo bản copy từ market-level dataset
# ------------------------------------------------------------

regime_df = market_level_df.copy()

regime_df = regime_df.sort_values("date").reset_index(drop=True)


# ------------------------------------------------------------
# 2. Chọn feature dùng để train HMM
# ------------------------------------------------------------

hmm_feature_cols = [
    "market_return",
    "market_volatility_20",
    "advance_decline_ratio",
    "percent_stocks_above_ma50",
    "percent_stocks_above_ma200",
    "avg_volume_change",
    "vix_close"
]

print("Các feature dùng cho HMM:")
print(hmm_feature_cols)


# ------------------------------------------------------------
# 3. Chuẩn bị dữ liệu cho HMM
# ------------------------------------------------------------

# Chỉ giữ các dòng có đủ feature để HMM học
hmm_data_df = regime_df[["date"] + hmm_feature_cols].dropna().reset_index(drop=True)

print("\nSố dòng market-level ban đầu:", regime_df.shape[0])
print("Số dòng dùng cho HMM:", hmm_data_df.shape[0])

# Tạo ma trận X cho HMM
X_hmm = hmm_data_df[hmm_feature_cols].values


# ------------------------------------------------------------
# 4. Scale dữ liệu trước khi đưa vào HMM
# ------------------------------------------------------------

hmm_scaler = StandardScaler()

X_hmm_scaled = hmm_scaler.fit_transform(X_hmm)


# ------------------------------------------------------------
# 5. Train Gaussian Hidden Markov Model
# ------------------------------------------------------------

n_regimes = 3

hmm_model = GaussianHMM(
    n_components=n_regimes,
    covariance_type="full",
    n_iter=1000,
    random_state=42
)

hmm_model.fit(X_hmm_scaled)

print("\nTrain Gaussian HMM hoàn tất.")
print("Số lượng regime:", n_regimes)


# ------------------------------------------------------------
# 6. Predict regime cho từng ngày
# ------------------------------------------------------------

hmm_regimes = hmm_model.predict(X_hmm_scaled)

hmm_data_df["regime"] = hmm_regimes


# ------------------------------------------------------------
# 7. Gắn regime ngược lại vào regime_df theo date
# ------------------------------------------------------------

regime_df = regime_df.merge(
    hmm_data_df[["date", "regime"]],
    on="date",
    how="left"
)

# regime sẽ bị NaN ở những dòng đầu thiếu rolling features
regime_df["regime"] = regime_df["regime"].astype("float32")


# ------------------------------------------------------------
# 8. Thống kê đặc điểm từng regime
# ------------------------------------------------------------

regime_summary_df = (
    regime_df
    .dropna(subset=["regime"])
    .groupby("regime")
    .agg(
        number_of_days=("date", "count"),
        avg_market_return=("market_return", "mean"),
        avg_market_volatility_20=("market_volatility_20", "mean"),
        avg_advance_decline_ratio=("advance_decline_ratio", "mean"),
        avg_percent_above_ma50=("percent_stocks_above_ma50", "mean"),
        avg_percent_above_ma200=("percent_stocks_above_ma200", "mean"),
        avg_vix_close=("vix_close", "mean")
    )
    .reset_index()
)

print("\n========== TỔNG KẾT REGIME ==========")
display(regime_summary_df)


# ------------------------------------------------------------
# 9. Gán ý nghĩa cho từng regime dựa trên thống kê
# ------------------------------------------------------------

# Vì HMM label là arbitrary, ta diễn giải dựa trên các đặc điểm:
# - Crisis / High volatility: VIX và volatility cao nhất
# - Bear / Negative return: market_return thấp nhất
# - Bull / Low volatility: market_return cao, VIX thấp, breadth tốt
# - Sideway / Neutral: regime còn lại

regime_interpretation = {}

summary_temp = regime_summary_df.copy()

# Regime có VIX trung bình cao nhất được xem là High volatility / Crisis
crisis_regime = summary_temp.sort_values(
    ["avg_vix_close", "avg_market_volatility_20"],
    ascending=False
)["regime"].iloc[0]

regime_interpretation[crisis_regime] = "High volatility / Crisis"

remaining_regimes = summary_temp[
    ~summary_temp["regime"].isin(regime_interpretation.keys())
]

# Trong các regime còn lại, regime có market_return thấp nhất là Bear
bear_regime = remaining_regimes.sort_values(
    "avg_market_return",
    ascending=True
)["regime"].iloc[0]

regime_interpretation[bear_regime] = "Bear / Negative return"

remaining_regimes = summary_temp[
    ~summary_temp["regime"].isin(regime_interpretation.keys())
]

# Trong các regime còn lại, regime có market_return cao nhất là Bull
bull_regime = remaining_regimes.sort_values(
    ["avg_market_return", "avg_advance_decline_ratio"],
    ascending=False
)["regime"].iloc[0]

regime_interpretation[bull_regime] = "Bull / Low volatility"

remaining_regimes = summary_temp[
    ~summary_temp["regime"].isin(regime_interpretation.keys())
]

# Regime còn lại là Sideway / Neutral
for regime_value in remaining_regimes["regime"].values:
    regime_interpretation[regime_value] = "Sideway / Neutral"


# Gắn tên regime vào dataset
regime_df["regime_name"] = regime_df["regime"].map(regime_interpretation)


print("\n========== DIỄN GIẢI REGIME ==========")
for regime_value, regime_name in regime_interpretation.items():
    print("Regime", int(regime_value), ":", regime_name)


# ------------------------------------------------------------
# 10. Kiểm tra dữ liệu sau khi gán regime
# ------------------------------------------------------------

print("\n========== TỔNG QUAN DỮ LIỆU REGIME ==========")
print("Kích thước regime dataset:", regime_df.shape)

print("\nSố dòng thiếu regime:")
print(regime_df["regime"].isnull().sum())

print("\nPhân phối regime:")
display(regime_df["regime_name"].value_counts(dropna=False))

print("\nMẫu dữ liệu regime:")
display(regime_df[[
    "date",
    "market_return",
    "market_volatility_20",
    "advance_decline_ratio",
    "percent_stocks_above_ma50",
    "percent_stocks_above_ma200",
    "vix_close",
    "regime",
    "regime_name"
]].head(20))


# ------------------------------------------------------------
# 11. Lưu lại market-level dataset có regime
# ------------------------------------------------------------

market_level_with_regime_file = output_path + "market_level_dataset.parquet"

regime_df.to_parquet(market_level_with_regime_file, index=False)

print("\nMarket-level dataset có regime đã lưu tại:")
print(market_level_with_regime_file)

print("\nGán nhãn regime hoàn tất thành công.")

In [ ]:
# ============================================================
# CELL 9: TẠO TARGET
# ============================================================

print("Bắt đầu tạo target...")


# ------------------------------------------------------------
# 1. Tạo bản copy từ regime_df
# ------------------------------------------------------------

modeling_df = regime_df.copy()

# Đảm bảo dữ liệu được sắp xếp theo thời gian
modeling_df = modeling_df.sort_values("date").reset_index(drop=True)


# ------------------------------------------------------------
# 2. Tạo target regime ngày tiếp theo
# ------------------------------------------------------------

modeling_df["target_regime_next_day"] = modeling_df["regime"].shift(-1)

modeling_df["target_regime_name_next_day"] = modeling_df["regime_name"].shift(-1)


# ------------------------------------------------------------
# 3. Xóa các dòng không dùng được cho modeling
# ------------------------------------------------------------

# Không dùng được nếu:
# - regime hiện tại bị thiếu
# - target ngày tiếp theo bị thiếu
modeling_df = modeling_df.dropna(
    subset=[
        "regime",
        "target_regime_next_day"
    ]
).reset_index(drop=True)


# ------------------------------------------------------------
# 4. Đổi target sang kiểu int
# ------------------------------------------------------------

modeling_df["regime"] = modeling_df["regime"].astype("int32")
modeling_df["target_regime_next_day"] = modeling_df["target_regime_next_day"].astype("int32")


# ------------------------------------------------------------
# 5. Kiểm tra target sau khi tạo
# ------------------------------------------------------------

print("\n========== TỔNG QUAN TẠO TARGET ==========")
print("Kích thước modeling dataset:", modeling_df.shape)

print("\nKhoảng thời gian:")
print("Ngày bắt đầu:", modeling_df["date"].min())
print("Ngày kết thúc:", modeling_df["date"].max())

print("\nPhân phối regime hiện tại:")
display(modeling_df["regime_name"].value_counts())

print("\nPhân phối target regime ngày tiếp theo:")
display(modeling_df["target_regime_name_next_day"].value_counts())

print("\nMẫu dữ liệu target:")
display(modeling_df[[
    "date",
    "market_return",
    "market_volatility_20",
    "advance_decline_ratio",
    "percent_stocks_above_ma50",
    "percent_stocks_above_ma200",
    "vix_close",
    "regime",
    "regime_name",
    "target_regime_next_day",
    "target_regime_name_next_day"
]].head(20))


print("\nTạo target hoàn tất thành công.")

In [ ]:
# ============================================================
# CELL 10: CHIA TRAIN / TEST THEO THỜI GIAN
# ============================================================

print("Bắt đầu chia train/test theo thời gian...")


# ------------------------------------------------------------
# 1. Tạo bản copy từ modeling_df
# ------------------------------------------------------------

model_data_df = modeling_df.copy()

model_data_df = model_data_df.sort_values("date").reset_index(drop=True)


# ------------------------------------------------------------
# 2. Chọn feature columns
# ------------------------------------------------------------

# Feature chính ở market-level
base_feature_cols = [
    "market_return",
    "market_volatility_20",
    "advance_decline_ratio",
    "percent_stocks_above_ma50",
    "percent_stocks_above_ma200",
    "avg_volume_change",
    "vix_close",
    "regime"
]

# Feature sector return
sector_feature_cols = [
    col for col in model_data_df.columns
    if col.startswith("sector_return_")
]

# Gộp toàn bộ feature
model_feature_cols = base_feature_cols + sector_feature_cols

print("Số lượng model features:", len(model_feature_cols))
print("\nCác model features:")
print(model_feature_cols)


# ------------------------------------------------------------
# 3. Chọn target column
# ------------------------------------------------------------

target_col = "target_regime_next_day"

print("\nCột target:", target_col)


# ------------------------------------------------------------
# 4. Xử lý missing values trước khi split
# ------------------------------------------------------------

# Chỉ giữ các dòng có đầy đủ feature và target
model_data_df = model_data_df.dropna(
    subset=model_feature_cols + [target_col]
).reset_index(drop=True)

print("\nKích thước model data sau khi xóa missing values:", model_data_df.shape)


# ------------------------------------------------------------
# 5. Tạo X và y
# ------------------------------------------------------------

X = model_data_df[model_feature_cols]
y = model_data_df[target_col]

dates = model_data_df["date"]


# ------------------------------------------------------------
# 6. Chia train/validation/test theo thời gian
# ------------------------------------------------------------

train_size = int(len(model_data_df) * 0.70)
validation_size = int(len(model_data_df) * 0.15)

train_end = train_size
validation_end = train_size + validation_size

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:validation_end]
X_test = X.iloc[validation_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:validation_end]
y_test = y.iloc[validation_end:]

dates_train = dates.iloc[:train_end]
dates_val = dates.iloc[train_end:validation_end]
dates_test = dates.iloc[validation_end:]


# ------------------------------------------------------------
# 7. Kiểm tra kết quả split
# ------------------------------------------------------------

print("\n========== TỔNG QUAN CHIA TRAIN / VALIDATION / TEST ==========")

print("Tổng số dòng:", len(model_data_df))
print("Số dòng train:", len(X_train))
print("Số dòng test:", len(X_test))
print("Số dòng validation:", len(X_val))

print("\nKhoảng thời gian train:")
print("Ngày bắt đầu:", dates_train.min())
print("Ngày kết thúc:", dates_train.max())

print("\nKhoảng thời gian test:")
print("Ngày bắt đầu:", dates_test.min())
print("Ngày kết thúc:", dates_test.max())

print("\nPhân phối target trong train:")
display(y_train.value_counts().sort_index())

print("\nPhân phối target trong test:")
display(y_test.value_counts().sort_index())

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nChia train/test hoàn tất thành công.")

In [ ]:
# ============================================================
# CELL 11: TRAIN MODELS
# ============================================================

print("Bắt đầu train models...")


# ------------------------------------------------------------
# 1. Import thêm công cụ cần thiết cho training
# ------------------------------------------------------------

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight


# ------------------------------------------------------------
# 2. Tạo dictionary để lưu model và prediction
# ------------------------------------------------------------

trained_models = {}
validation_predictions = {}
model_predictions = {}


# ------------------------------------------------------------
# 3. Train Logistic Regression baseline model
# ------------------------------------------------------------

logistic_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

y_pred_logistic_val = logistic_model.predict(X_val)
y_pred_logistic = logistic_model.predict(X_test)

trained_models["Logistic Regression"] = logistic_model
validation_predictions["Logistic Regression"] = y_pred_logistic_val
model_predictions["Logistic Regression"] = y_pred_logistic


print("Train Logistic Regression hoàn tất.")


# ------------------------------------------------------------
# 4. Train Random Forest Classifier
# ------------------------------------------------------------

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

y_pred_random_forest_val = random_forest_model.predict(X_val)
y_pred_random_forest = random_forest_model.predict(X_test)

trained_models["Random Forest"] = random_forest_model
validation_predictions["Random Forest"] = y_pred_random_forest_val
model_predictions["Random Forest"] = y_pred_random_forest

print("Train Random Forest hoàn tất.")


# ------------------------------------------------------------
# 5. Chuẩn bị label encoding cho XGBoost
# ------------------------------------------------------------

# XGBoost hoạt động ổn định hơn khi label là 0, 1, 2, ...
xgb_label_encoder = LabelEncoder()

y_train_xgb = xgb_label_encoder.fit_transform(y_train)

print("\nMapping class cho XGBoost:")
for encoded_label, original_label in enumerate(xgb_label_encoder.classes_):
    print("Encoded", encoded_label, "-> regime gốc", original_label)


# ------------------------------------------------------------
# 6. Tính sample weight cho XGBoost để giảm ảnh hưởng mất cân bằng class
# ------------------------------------------------------------

xgb_sample_weight = compute_sample_weight(
    class_weight="balanced",
    y=y_train_xgb
)


# ------------------------------------------------------------
# 7. Train XGBoost Classifier
# ------------------------------------------------------------

xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgboost_model.fit(
    X_train,
    y_train_xgb,
    sample_weight=xgb_sample_weight
)

y_pred_xgb_val_encoded = xgboost_model.predict(X_val)
y_pred_xgb_encoded = xgboost_model.predict(X_test)

y_pred_xgboost_val = xgb_label_encoder.inverse_transform(y_pred_xgb_val_encoded)
y_pred_xgboost = xgb_label_encoder.inverse_transform(y_pred_xgb_encoded)

trained_models["XGBoost"] = xgboost_model
validation_predictions["XGBoost"] = y_pred_xgboost_val
model_predictions["XGBoost"] = y_pred_xgboost

print("Train XGBoost hoàn tất.")


# ------------------------------------------------------------
# 8. Lưu XGBoost model và label encoder
# ------------------------------------------------------------

xgboost_model_file = model_path + "xgboost_model.pkl"
xgboost_label_encoder_file = model_path + "xgboost_label_encoder.pkl"

joblib.dump(xgboost_model, xgboost_model_file)
joblib.dump(xgb_label_encoder, xgboost_label_encoder_file)

print("\nXGBoost model đã lưu tại:")
print(xgboost_model_file)

print("\nXGBoost label encoder đã lưu tại:")
print(xgboost_label_encoder_file)


# ------------------------------------------------------------
# 9. Kiểm tra nhanh prediction của từng model
# ------------------------------------------------------------

print("\n========== TỔNG QUAN TRAINING MODEL ==========")

for model_name, prediction in model_predictions.items():
    print("\nModel:", model_name)
    print("Kích thước prediction:", prediction.shape)
    print("Phân phối class dự đoán:")
    display(pd.Series(prediction).value_counts().sort_index())


print("\nPhân phối target thực tế trong test:")
display(y_test.value_counts().sort_index())

print("\nTraining model hoàn tất thành công.")

In [ ]:
# ============================================================
# CELL 12: MODEL EVALUATION WITH VALIDATION AND TEST SET
# ============================================================

print("Start model evaluation with validation and test set...")


# ------------------------------------------------------------
# 1. Tạo mapping regime -> regime_name để dễ đọc kết quả
# ------------------------------------------------------------

regime_name_map = (
    regime_df
    .dropna(subset=["regime", "regime_name"])
    [["regime", "regime_name"]]
    .drop_duplicates()
)

regime_name_map["regime"] = regime_name_map["regime"].astype("int32")

regime_name_dict = dict(
    zip(
        regime_name_map["regime"],
        regime_name_map["regime_name"]
    )
)

print("Regime name mapping:")
for regime_value, regime_name in regime_name_dict.items():
    print("Regime", regime_value, ":", regime_name)


# ------------------------------------------------------------
# 2. Đánh giá trên validation set để chọn model tốt nhất
# ------------------------------------------------------------

validation_results = []

print("\n========== VALIDATION SET EVALUATION ==========")

for model_name, y_pred_val in validation_predictions.items():

    print("\n" + "=" * 80)
    print("VALIDATION MODEL:", model_name)
    print("=" * 80)

    y_pred_val = pd.Series(y_pred_val).astype("int32")
    y_true_val = y_val.reset_index(drop=True).astype("int32")

    labels = sorted(list(set(y_true_val.unique()) | set(y_pred_val.unique())))

    target_names = [
        "Regime " + str(label) + " - " + regime_name_dict.get(label, "Unknown")
        for label in labels
    ]

    accuracy = accuracy_score(y_true_val, y_pred_val)

    precision_macro = precision_score(
        y_true_val,
        y_pred_val,
        labels=labels,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_true_val,
        y_pred_val,
        labels=labels,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_true_val,
        y_pred_val,
        labels=labels,
        average="macro",
        zero_division=0
    )

    precision_weighted = precision_score(
        y_true_val,
        y_pred_val,
        labels=labels,
        average="weighted",
        zero_division=0
    )

    recall_weighted = recall_score(
        y_true_val,
        y_pred_val,
        labels=labels,
        average="weighted",
        zero_division=0
    )

    f1_weighted = f1_score(
        y_true_val,
        y_pred_val,
        labels=labels,
        average="weighted",
        zero_division=0
    )

    validation_results.append({
        "model": model_name,
        "validation_accuracy": accuracy,
        "validation_precision_macro": precision_macro,
        "validation_recall_macro": recall_macro,
        "validation_f1_macro": f1_macro,
        "validation_precision_weighted": precision_weighted,
        "validation_recall_weighted": recall_weighted,
        "validation_f1_weighted": f1_weighted
    })

    print("\nValidation metrics:")
    print("Accuracy:", round(accuracy, 4))
    print("Precision macro:", round(precision_macro, 4))
    print("Recall macro:", round(recall_macro, 4))
    print("F1 macro:", round(f1_macro, 4))
    print("Precision weighted:", round(precision_weighted, 4))
    print("Recall weighted:", round(recall_weighted, 4))
    print("F1 weighted:", round(f1_weighted, 4))

    print("\nValidation classification report:")
    print(classification_report(
        y_true_val,
        y_pred_val,
        labels=labels,
        target_names=target_names,
        zero_division=0
    ))


validation_comparison_df = pd.DataFrame(validation_results)

validation_comparison_df = validation_comparison_df.sort_values(
    by="validation_f1_macro",
    ascending=False
).reset_index(drop=True)

print("\n========== VALIDATION MODEL COMPARISON ==========")
display(validation_comparison_df)


# ------------------------------------------------------------
# 3. Chọn model tốt nhất dựa trên validation F1 macro
# ------------------------------------------------------------

best_model_name = validation_comparison_df.loc[0, "model"]
best_validation_f1_macro = validation_comparison_df.loc[0, "validation_f1_macro"]

print("\nBest model based on validation F1 macro:")
print(best_model_name)
print("Validation F1 macro:", round(best_validation_f1_macro, 4))


# ------------------------------------------------------------
# 4. Đánh giá cuối cùng trên test set
# ------------------------------------------------------------

evaluation_results = []
classification_reports = {}
confusion_matrices = {}

print("\n========== TEST SET EVALUATION ==========")

for model_name, y_pred in model_predictions.items():

    print("\n" + "=" * 80)
    print("TEST MODEL:", model_name)
    print("=" * 80)

    y_pred = pd.Series(y_pred).astype("int32")
    y_true = y_test.reset_index(drop=True).astype("int32")

    labels = sorted(list(set(y_true.unique()) | set(y_pred.unique())))

    target_names = [
        "Regime " + str(label) + " - " + regime_name_dict.get(label, "Unknown")
        for label in labels
    ]

    accuracy = accuracy_score(y_true, y_pred)

    precision_macro = precision_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

    precision_weighted = precision_score(
        y_true,
        y_pred,
        labels=labels,
        average="weighted",
        zero_division=0
    )

    recall_weighted = recall_score(
        y_true,
        y_pred,
        labels=labels,
        average="weighted",
        zero_division=0
    )

    f1_weighted = f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="weighted",
        zero_division=0
    )

    evaluation_results.append({
        "model": model_name,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    })

    print("\nTest metrics:")
    print("Accuracy:", round(accuracy, 4))
    print("Precision macro:", round(precision_macro, 4))
    print("Recall macro:", round(recall_macro, 4))
    print("F1 macro:", round(f1_macro, 4))
    print("Precision weighted:", round(precision_weighted, 4))
    print("Recall weighted:", round(recall_weighted, 4))
    print("F1 weighted:", round(f1_weighted, 4))

    report_text = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        zero_division=0
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        zero_division=0,
        output_dict=True
    )

    classification_reports[model_name] = report_dict

    print("\nTest classification report:")
    print(report_text)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=labels
    )

    cm_df = pd.DataFrame(
        cm,
        index=["Actual " + str(label) for label in labels],
        columns=["Predicted " + str(label) for label in labels]
    )

    confusion_matrices[model_name] = cm_df

    print("\nTest confusion matrix:")
    display(cm_df)


# ------------------------------------------------------------
# 5. Tạo bảng so sánh model trên test set
# ------------------------------------------------------------

model_comparison_df = pd.DataFrame(evaluation_results)

model_comparison_df = model_comparison_df.sort_values(
    by="f1_macro",
    ascending=False
).reset_index(drop=True)

print("\n========== TEST MODEL COMPARISON ==========")
display(model_comparison_df)


# ------------------------------------------------------------
# 6. Hiển thị kết quả model tốt nhất theo validation trên test set
# ------------------------------------------------------------

best_model_test_result = model_comparison_df[
    model_comparison_df["model"] == best_model_name
]

print("\nBest model selected by validation:")
print(best_model_name)

print("\nTest result of selected best model:")
display(best_model_test_result)


# ------------------------------------------------------------
# 7. Lưu kết quả comparison
# ------------------------------------------------------------

validation_comparison_file = output_path + "validation_model_comparison.csv"
model_comparison_file = output_path + "model_comparison.csv"

validation_comparison_df.to_csv(validation_comparison_file, index=False)
model_comparison_df.to_csv(model_comparison_file, index=False)

print("\nValidation model comparison saved to:")
print(validation_comparison_file)

print("\nTest model comparison saved to:")
print(model_comparison_file)

print("\nModel evaluation with validation and test set completed successfully.")

In [ ]:
# ============================================================
# CELL 13: TRỰC QUAN HÓA
# ============================================================

print("Bắt đầu trực quan hóa...")


# ------------------------------------------------------------
# 1. Chuẩn bị dữ liệu visualization
# ------------------------------------------------------------

viz_df = regime_df.copy()

viz_df = viz_df.sort_values("date").reset_index(drop=True)

# Chỉ giữ dòng có regime hợp lệ để vẽ
viz_df = viz_df.dropna(subset=["regime", "regime_name"]).reset_index(drop=True)

viz_df["regime"] = viz_df["regime"].astype("int32")

# Tạo cumulative market return để mô phỏng đường thị trường
viz_df["cumulative_market_return"] = (1 + viz_df["market_return"].fillna(0)).cumprod()

print("Kích thước dataset dùng để trực quan hóa:", viz_df.shape)

print("\nPhân phối regime dùng để trực quan hóa:")
display(viz_df["regime_name"].value_counts())


# ------------------------------------------------------------
# 2. Regime chart: market return / cumulative return theo thời gian
# ------------------------------------------------------------

plt.figure(figsize=(16, 7))

# Vẽ đường cumulative market return
plt.plot(
    viz_df["date"],
    viz_df["cumulative_market_return"],
    linewidth=1.5,
    label="Cumulative Market Return"
)

# Tô điểm theo từng regime
for regime_name in viz_df["regime_name"].unique():
    regime_part = viz_df[viz_df["regime_name"] == regime_name]

    plt.scatter(
        regime_part["date"],
        regime_part["cumulative_market_return"],
        s=12,
        label=regime_name,
        alpha=0.7
    )

plt.title("Market Regime theo thời gian")
plt.xlabel("Date")
plt.ylabel("Cumulative Market Return")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

regime_chart_file = chart_path + "regime_chart.png"
plt.savefig(regime_chart_file, dpi=300)
plt.show()

print("Regime chart đã lưu tại:")
print(regime_chart_file)


# ------------------------------------------------------------
# 3. Confusion matrix của model tốt nhất
# ------------------------------------------------------------

best_y_pred = pd.Series(model_predictions[best_model_name]).astype("int32")
best_y_true = y_test.reset_index(drop=True).astype("int32")

best_labels = sorted(list(set(best_y_true.unique()) | set(best_y_pred.unique())))

best_cm = confusion_matrix(
    best_y_true,
    best_y_pred,
    labels=best_labels
)

best_cm_labels = [
    "Regime " + str(label) + "\n" + regime_name_dict.get(label, "Unknown")
    for label in best_labels
]

plt.figure(figsize=(9, 7))

sns.heatmap(
    best_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=best_cm_labels,
    yticklabels=best_cm_labels
)

plt.title("Confusion Matrix - " + best_model_name)
plt.xlabel("Nhãn dự đoán")
plt.ylabel("Nhãn thực tế")
plt.tight_layout()

confusion_matrix_file = chart_path + "confusion_matrix.png"
plt.savefig(confusion_matrix_file, dpi=300)
plt.show()

print("Confusion matrix đã lưu tại:")
print(confusion_matrix_file)


# ------------------------------------------------------------
# 4. Bảng so sánh model chart
# ------------------------------------------------------------

model_comparison_plot_df = model_comparison_df.copy()

plt.figure(figsize=(10, 6))

sns.barplot(
    data=model_comparison_plot_df,
    x="model",
    y="f1_macro"
)

plt.title("So sánh model theo F1 Macro")
plt.xlabel("Model")
plt.ylabel("F1 Macro")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

model_comparison_chart_file = chart_path + "model_comparison_chart.png"
plt.savefig(model_comparison_chart_file, dpi=300)
plt.show()

print("Bảng so sánh model chart đã lưu tại:")
print(model_comparison_chart_file)


# ------------------------------------------------------------
# 5. Feature importance của Random Forest
# ------------------------------------------------------------

random_forest_importance_df = pd.DataFrame({
    "feature": model_feature_cols,
    "importance": trained_models["Random Forest"].feature_importances_
})

random_forest_importance_df = random_forest_importance_df.sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)

top_rf_features = random_forest_importance_df.head(20)

plt.figure(figsize=(10, 8))

sns.barplot(
    data=top_rf_features,
    x="importance",
    y="feature"
)

plt.title("Top 20 Feature Importance - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()

rf_feature_importance_file = chart_path + "random_forest_feature_importance.png"
plt.savefig(rf_feature_importance_file, dpi=300)
plt.show()

print("Random Forest feature importance chart đã lưu tại:")
print(rf_feature_importance_file)

print("\nTop Random Forest features:")
display(top_rf_features)


# ------------------------------------------------------------
# 6. Feature importance của XGBoost
# ------------------------------------------------------------

xgboost_importance_df = pd.DataFrame({
    "feature": model_feature_cols,
    "importance": trained_models["XGBoost"].feature_importances_
})

xgboost_importance_df = xgboost_importance_df.sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)

top_xgb_features = xgboost_importance_df.head(20)

plt.figure(figsize=(10, 8))

sns.barplot(
    data=top_xgb_features,
    x="importance",
    y="feature"
)

plt.title("Top 20 Feature Importance - XGBoost")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()

xgb_feature_importance_file = chart_path + "xgboost_feature_importance.png"
plt.savefig(xgb_feature_importance_file, dpi=300)
plt.show()

print("XGBoost feature importance chart đã lưu tại:")
print(xgb_feature_importance_file)

print("\nTop XGBoost features:")
display(top_xgb_features)


# ------------------------------------------------------------
# 7. VIX theo regime
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

sns.boxplot(
    data=viz_df,
    x="regime_name",
    y="vix_close"
)

plt.title("Phân phối VIX theo Market Regime")
plt.xlabel("Market Regime")
plt.ylabel("VIX Close")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

vix_by_regime_file = chart_path + "vix_by_regime.png"
plt.savefig(vix_by_regime_file, dpi=300)
plt.show()

print("Chart VIX theo regime đã lưu tại:")
print(vix_by_regime_file)


# ------------------------------------------------------------
# 8. Market return theo regime
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

sns.boxplot(
    data=viz_df,
    x="regime_name",
    y="market_return"
)

plt.title("Phân phối Market Return theo Market Regime")
plt.xlabel("Market Regime")
plt.ylabel("Market Return")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

market_return_by_regime_file = chart_path + "market_return_by_regime.png"
plt.savefig(market_return_by_regime_file, dpi=300)
plt.show()

print("Chart market return theo regime đã lưu tại:")
print(market_return_by_regime_file)


# ------------------------------------------------------------
# 9. Tổng kết file chart đã lưu
# ------------------------------------------------------------

print("\n========== CÁC FILE CHART ĐÃ LƯU ==========")
print("1.", regime_chart_file)
print("2.", confusion_matrix_file)
print("3.", model_comparison_chart_file)
print("4.", rf_feature_importance_file)
print("5.", xgb_feature_importance_file)
print("6.", vix_by_regime_file)
print("7.", market_return_by_regime_file)

print("\nTrực quan hóa hoàn tất thành công.")

In [ ]:
# ============================================================
# CELL 14: LƯU OUTPUT CUỐI VÀ TỔNG KẾT
# ============================================================

print("Bắt đầu kiểm tra output cuối và tổng kết...")


# ------------------------------------------------------------
# 1. Khai báo lại danh sách file output cần kiểm tra
# ------------------------------------------------------------

required_output_files = {
    "Stock data sau cleaning": output_path + "stock_clean.parquet",
    "Stock-level features": output_path + "stock_features.parquet",
    "Market-level dataset có regime": output_path + "market_level_dataset.parquet",
    "Bảng so sánh model": output_path + "model_comparison.csv",
    "Regime chart": chart_path + "regime_chart.png",
    "Confusion matrix chart": chart_path + "confusion_matrix.png",
    "XGBoost model": model_path + "xgboost_model.pkl"
}


# ------------------------------------------------------------
# 2. Kiểm tra file có tồn tại không
# ------------------------------------------------------------

file_check_results = []

for file_name, file_path in required_output_files.items():
    file_exists = os.path.exists(file_path)

    if file_exists:
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    else:
        file_size_mb = 0

    file_check_results.append({
        "file_name": file_name,
        "file_path": file_path,
        "exists": file_exists,
        "file_size_mb": round(file_size_mb, 4)
    })

file_check_df = pd.DataFrame(file_check_results)

print("\n========== KIỂM TRA FILE OUTPUT CUỐI ==========")
display(file_check_df)


# ------------------------------------------------------------
# 3. Tổng kết dataset sau xử lý
# ------------------------------------------------------------

final_data_summary = {
    "stock_clean_rows": stock_df.shape[0],
    "stock_clean_columns": stock_df.shape[1],
    "stock_features_rows": stock_features_df.shape[0],
    "stock_features_columns": stock_features_df.shape[1],
    "market_level_rows": market_level_df.shape[0],
    "market_level_columns": market_level_df.shape[1],
    "regime_dataset_rows": regime_df.shape[0],
    "modeling_rows": modeling_df.shape[0],
    "number_of_tickers": stock_df["ticker"].nunique(),
    "number_of_regimes": regime_df["regime"].nunique(),
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "test_rows": len(X_test)
}

final_data_summary_df = pd.DataFrame(
    list(final_data_summary.items()),
    columns=["item", "value"]
)

print("\n========== TỔNG KẾT DỮ LIỆU CUỐI ==========")
display(final_data_summary_df)


# ------------------------------------------------------------
# 4. Tổng kết regime
# ------------------------------------------------------------

final_regime_summary_df = (
    regime_df
    .dropna(subset=["regime", "regime_name"])
    .groupby(["regime", "regime_name"])
    .agg(
        number_of_days=("date", "count"),
        avg_market_return=("market_return", "mean"),
        avg_market_volatility_20=("market_volatility_20", "mean"),
        avg_vix_close=("vix_close", "mean"),
        avg_advance_decline_ratio=("advance_decline_ratio", "mean")
    )
    .reset_index()
)

print("\n========== TỔNG KẾT REGIME CUỐI ==========")
display(final_regime_summary_df)


# ------------------------------------------------------------
# 5. Tổng kết model comparison
# ------------------------------------------------------------

print("\n========== SO SÁNH MODEL CUỐI ==========")
display(model_comparison_df)

print("\nModel tốt nhất dựa trên F1 macro:")
print(best_model_name)


# ------------------------------------------------------------
# 6. In kết luận ngắn cho notebook
# ------------------------------------------------------------

print("\n========== TỔNG KẾT ĐỒ ÁN ==========")

print("Tên đề tài: Forecasting Financial Market Regimes")
print("Dataset chính: dữ liệu OHLCV lịch sử của cổ phiếu S&P 500")
print("Dataset phụ: sector metadata của S&P 500 và chỉ số VIX")
print("Model gán nhãn regime: Gaussian Hidden Markov Model")
print("Các model dự đoán: Logistic Regression, Random Forest, XGBoost")
print("Target dự đoán chính: target_regime_next_day")
print("Model tốt nhất dựa trên F1 macro:", best_model_name)

print("\nTrọng tâm:")
print("- Xử lý dữ liệu OHLCV daily của nhiều cổ phiếu S&P 500")
print("- Làm sạch missing values, dòng trùng lặp, cột ngày và cột numeric")
print("- Join stock data với sector metadata và VIX data")
print("- Tạo technical features ở cấp cổ phiếu")
print("- Tổng hợp stock-level data thành market-level dataset")
print("- Dùng HMM để gán nhãn market regimes")
print("- Train machine learning models để dự đoán market regime ngày tiếp theo")

print("\nKiểm tra output cuối hoàn tất thành công.")